# 3. Variational inference and the Rényi objective

Use `BayesianLinear`, `GaussianLikelihood`, and `VariationalLoss` to learn a
regression model with stochastic activations and an observation-noise scale.
See [setup and running instructions](README.md). This is a CPU-only synthetic example.

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
from torch import nn

from bensemble.layers import BayesianLinear
from bensemble.losses import GaussianLikelihood, VariationalLoss
from bensemble.utils import get_total_kl, predict_with_uncertainty

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)  # Small CPU workloads avoid thread-pool overhead.
DEVICE = torch.device("cpu")
ALPHA = 1.0  # Set to 0.5 and rerun all cells to train with the Rényi objective.
MC_SAMPLES = 4
EPOCHS = 800

## 1. Data and evaluation protocol

Generate a noisy regression problem and split it into disjoint train, validation,
and test sets. Training inputs lie in `[-3, 3]`; the plotting grid extends to
`[-5, 5]`. Test RMSE measures interpolation on fresh noisy targets. The wider grid
illustrates extrapolation, without assuming that uncertainty must grow outside
the training range.

The validation split is reserved for any hyperparameter tuning; this fixed-configuration run does not use it.

In [ ]:
def true_function(x):
    return torch.sin(1.5 * x) + 0.2 * x


generator = torch.Generator().manual_seed(SEED)
features = 6 * torch.rand(300, 1, generator=generator) - 3
targets = true_function(features) + 0.15 * torch.randn(300, 1, generator=generator)
indices = torch.randperm(len(features), generator=generator)
split_indices = {
    "train": indices[:180],
    "validation": indices[180:240],
    "test": indices[240:],
}
x_train, y_train = features[split_indices["train"]], targets[split_indices["train"]]
x_val, y_val = (
    features[split_indices["validation"]],
    targets[split_indices["validation"]],
)
x_test, y_test = features[split_indices["test"]], targets[split_indices["test"]]
x_grid = torch.linspace(-5, 5, 240).reshape(-1, 1)

plt.figure(figsize=(8, 4))
plt.scatter(x_train[:, 0], y_train[:, 0], s=12, label="Train")
plt.plot(x_grid[:, 0], true_function(x_grid)[:, 0], "k--", label="True function")
plt.axvspan(-3, 3, alpha=0.08, color="green", label="Training input range")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

## 2. Define the objective

Bayesian layers use local reparameterization: during training they sample
activations from the distributions induced by uncertain weights. In evaluation
mode their ordinary forward pass is deterministic.

The likelihood learns one global noise standard deviation. We train on the whole
training split in each step, so `num_batches=1` and `kl_weight=1.0` apply the full KL
penalty. With minibatches, set `num_batches` to the number of training batches.

In [ ]:
model = nn.Sequential(BayesianLinear(1, 32), nn.Tanh(), BayesianLinear(32, 1)).to(
    DEVICE
)
likelihood = GaussianLikelihood(init_log_sigma=-1.5).to(DEVICE)
criterion = VariationalLoss(likelihood, alpha=ALPHA, num_batches=1, kl_weight=1.0)
optimizer = torch.optim.Adam(
    list(model.parameters()) + list(likelihood.parameters()), lr=0.01
)

## 3. Train with multiple stochastic forward passes

Stack predictions along a new leading axis: `(MC samples, batch, outputs)`.
`alpha=1` gives the ELBO objective; `alpha != 1` uses the library's Rényi-style
objective. **Keep multiple samples when changing alpha**: with one sample the
log-mean-exp collapses and changing alpha does not change the objective.

In [ ]:
model.train()
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    predictions = torch.stack([model(x_train) for _ in range(MC_SAMPLES)])
    loss = criterion(predictions, y_train, get_total_kl(model))
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 200 == 0:
        print(
            f"Epoch {epoch + 1}: loss={loss.item():.2f}, noise std={likelihood.sigma:.3f}"
        )

# Evaluate both objectives on the same stochastic draws to expose alpha's effect.
with torch.no_grad():
    draws = torch.stack([model(x_train) for _ in range(MC_SAMPLES)])
    kl = get_total_kl(model)
    elbo_loss = VariationalLoss(likelihood, alpha=1.0)(draws, y_train, kl)
    renyi_loss = VariationalLoss(likelihood, alpha=0.5)(draws, y_train, kl)
print(f"Same draws: ELBO={elbo_loss.item():.3f}, Rényi={renyi_loss.item():.3f}")

## 4. Combine model variation and observation noise

`predict_with_uncertainty` enables Bayesian sampling internally, even after
`model.eval()`. Add the learned observation variance to the variance of model
predictions. The resulting mean ± 2 standard deviations is a moment-based summary;
it does not establish 95% coverage for the predictive mixture or for shifted data.

In [ ]:
model.eval()
mean, std_model = predict_with_uncertainty(model, x_grid, num_samples=100)
test_mean, _ = predict_with_uncertainty(model, x_test, num_samples=100)
std_total = torch.sqrt(std_model.square() + likelihood.sigma**2)
assert mean.shape == std_total.shape == x_grid.shape
assert torch.isfinite(std_total).all()
assert std_model.max() > 0
test_rmse = (test_mean - y_test).square().mean().sqrt().item()
print(f"Held-out test RMSE: {test_rmse:.3f}")

plt.figure(figsize=(9, 4))
plt.scatter(x_train[:, 0], y_train[:, 0], s=12, alpha=0.5, label="Train")
plt.plot(x_grid[:, 0], true_function(x_grid)[:, 0], "k--", label="True function")
plt.plot(x_grid[:, 0], mean[:, 0], label="Predictive mean")
for std, label, opacity in [
    (std_total, "Mean ± 2 total standard deviations", 0.15),
    (std_model, "Mean ± 2 model standard deviations", 0.3),
]:
    plt.fill_between(
        x_grid[:, 0],
        (mean - 2 * std)[:, 0],
        (mean + 2 * std)[:, 0],
        alpha=opacity,
        label=label,
    )
plt.axvline(-3, color="grey", linestyle=":")
plt.axvline(3, color="grey", linestyle=":")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

Next: [Laplace approximation](04_laplace.ipynb) samples models around an already trained network.